# 课程 4 · 让真实模型处理第一封外贸询盘

上一课，你已经学过怎样组织 `messages`、从 API 响应里取出正文，以及为什么“格式正确”不等于“内容正确”。当时用过人工返回样例，还没有验证真实模型的判断。

这一次，我们为一家**虚构的保温杯出口工厂——山行杯业**做销售助手。它在国际站接到海外客户的询价，业务员要查商品、核对数量和时间，再准备回复。我们先完成其中一段：**收到一封询盘后，给出有依据的初步接单建议。**

## 开始时，我们有什么

| 手里的东西 | 来源和用途 |
|---|---|
| 买家发来的英文询盘 | 课程模拟；作为客户原话 |
| 工厂的一页资料 | 课程模拟的商品、库存、价格、定制和运输规则；作为本课事实依据 |
| 你的 Python 环境与模型账号 | 调用 DeepSeek 或腾讯云 TokenHub，获得真实生成的回答 |
| 一份人工核对要求 | 检查模型有没有漏问、乱承诺、忽略交期冲突 |

业务资料是 mock（我们编制的模拟资料），**三次回答都来自真实模型**。课程没有预先写好的模型答案，也没有调用失败后改读样例的开关。国际站消息收发暂时由这封本地询盘代替，今天的产物显示在笔记本里，交给业务员核对。

## 今天会看到什么

1. 只给买家询盘：模型能理解客户，但不知道这家工厂能做什么。
2. 加上工厂资料：看看它能否找到可行商品，并说明报价和交期的条件。
3. 把到货要求改成 10 天：看看它是否发现冲突，并调整回复。

主实验共 **3 次请求**，每运行一个调用单元都会再请求一次；先逐格运行。首次配置会隐藏输入 API key，请不要把 key 填进代码或聊天。每次输出有上限，服务仍按账号规则计费；上限不是总费用预算。

代码实验约 20–30 分钟，阅读和讨论按你的节奏来。本课的核心问题只有一个：**模型怎样拿到判断一笔订单所需的依据？**


## 1 · 先让程序拿到买家原话

选择你界面里的 **`llm-from-zero` 内核**。名字只是标签，下面会打印实际 Python 路径；项目环境通常以 `llm-from-zero/.venv/bin/python` 结尾。

下面这些准备代码已经写好：定位课程文件、导入发送请求的函数、读取模拟询盘。`Path.read_text()` 读出文件文字，`json.loads()` 把 JSON 文字读成字典——后者上一课用过。路径处理无需填空。


In [6]:
from pathlib import Path
import sys
import json
from IPython.display import display, Markdown

current_dir = Path.cwd()
project_dir = next((p for p in [current_dir, *current_dir.parents] if (p / "pyproject.toml").exists()), None)
assert project_dir is not None, "请从 llm-from-zero 项目或它的课程文件夹打开笔记本。"
course_dir = project_dir / "04-trade-inquiry"
if str(course_dir) not in sys.path:
    sys.path.insert(0, str(course_dir))
from trade_client import configure, call_model

inquiries = json.loads((course_dir / "data" / "inquiries.json").read_text(encoding="utf-8"))
print("Python：", sys.executable)
print("资料位置：", course_dir / "data")
print("本格只准备文件和函数，还没有请求模型。")


Python： /Users/yorkchen/文稿 - 陈尧的MacBook Pro/VibeCoding/llm-from-zero/.venv/bin/python
资料位置： /Users/yorkchen/文稿 - 陈尧的MacBook Pro/VibeCoding/llm-from-zero/04-trade-inquiry/data
本格只准备文件和函数，还没有请求模型。


In [7]:
buyer_text = inquiries["initial"]
print(buyer_text)


Hello, we are looking for 500 stainless steel vacuum bottles, 500ml each, with our company logo printed in one color. Your standard individual paper boxes are fine. We need them delivered to Germany next month. Which products would you recommend, and could you send us a quotation?


买家的意思是：想买 **500 个 500ml 保温杯，印一个单色 Logo，用普通纸盒，下个月送到德国**，请我们推荐颜色并给出价格和交期。

先站在业务员的位置想一想：只看这段话，你知道我们有哪些颜色、多少库存、能不能印 Logo、运费多少吗？客户说“下个月”，也还没说具体哪天要收到。

我们先把同样的问题交给真实模型。先不给工厂资料，观察它怎样处理不知道的信息。


## 2 · 第一次真正连接模型

`configure()` 准备调用凭证，**这一步不发送询盘**。`call_model(messages, config)` 才会发送一次请求。

- `PROVIDER` 选 `"deepseek"` 或 `"tokenhub"`。
- `MODEL = None` 时，DeepSeek 读取 `DEEPSEEK_MODEL`，未设置则使用已核实的默认模型 `deepseek-flash`；TokenHub 读取 `TOKENHUB_MODEL`，未设置则提示输入账号已开通的模型 ID。你也可以把模型 ID 直接写在 `MODEL` 后面，模型名不是密钥。
- 本课的 TokenHub 使用按量 API，暂只适配 DeepSeek 系列；默认广州地域，新加坡地域的配置方式见 README。
- 有相应环境变量时会直接读取；没有 key 时会弹出隐藏输入。粘贴后按回车，看不到字符是正常的。

请求发送和报错处理放在 [trade_client.py](trade_client.py) 中。今天先理解发了什么、收到什么，通信代码可以以后再细看。具体入口和模型 ID 的获取方式见 [README](README.md)。


In [13]:
PROVIDER = "deepseek"  # 也可选 "tokenhub"
MODEL = "deepseek-flash"  # 有需要可填账号中的模型 ID
config = configure(provider=PROVIDER, model=MODEL)
print(config)  # 配置对象的显示内容不包含 API key


ValueError: API key 中出现了空白或非 ASCII 字符，请重新输入完整密钥。

现在写任务说明。这里有两种不同的材料：**任务说明规定助手怎样工作，买家原话说明这次客户想要什么。** `system` 和 `user` 的写法沿用上一课。

下面的任务说明不含任何工厂库存或价格。先读一遍，看看“缺信息就追问”的要求是否清楚。本课先让业务员阅读文字建议；以后程序需要按字段处理时，再使用上一课的 JSON。


In [ ]:
task_rules = """你是山行杯业的外贸销售助理，帮助业务员准备接单建议。
只依据本次提供的买家原话和工厂资料判断；缺少的信息明确说不知道并追问。
买家提出的要求不等于工厂具备该能力，不能当成工厂规则。
用中文给业务员写三个部分：1. 已知需求；2. 初步建议和依据；3. 需要确认的问题。
再给一段不超过80个英文单词的回复草稿。中文部分控制在350字左右。
报价要说清包含什么、没有包含什么；交期估计和承诺要分清。
没有足够依据就不承诺价格、库存或到货日期。不要声称已联系客户或已下单。"""
first_messages = [
    {"role": "system", "content": task_rules},
    {"role": "user", "content": buyer_text},
]
print("准备发出", len(first_messages), "条消息；尚未调用模型。")


准备发出 2 条消息；尚未调用模型。


**下面这一格发出第 1 次真实请求。** 只运行一次，等待返回。失败会显示具体问题；不要用一段手写回答代替这次结果继续比较。

`first_result` 是函数整理后的返回值。`first_result["text"]` 是回答正文；这里的 `text` 是我们给工具函数约定的名字，**不是服务商原始响应新增了一个字段**。工具内部仍从 `choices[0]["message"]["content"]` 取正文，和上一课的取法相同。


In [10]:
first_result = call_model(first_messages, config)
print(first_result["text"])


**1. 已知需求**
买家要500只不锈钢真空瓶，500ml，单色印公司logo，接受标准独立纸盒，交德国，时间“下个月”。未说明瓶型、材质等级、印刷位置、包装文字、贸易术语和目的港。

**2. 初步建议和依据**
可先按工厂现有500ml不锈钢真空瓶系列推荐，优先选起订量低、可单色丝印或移印的常规款；依据是买家接受标准纸盒且数量500只。报价需列明：瓶体、单色logo印刷、标准独立纸盒、常规外箱；未含运费、关税、德国目的港费用及可能的打样/版费。交期只能给“估计”，不能承诺下月到货；需先确认库存和排产。

**3. 需要确认的问题**
请确认具体瓶型/颜色、logo文件与印刷位置、是否要打样、贸易术语（FOB/CIF等）及德国目的港、外箱唛头要求。另需工厂确认500ml常规款500只的起订量、单色印刷费和当前排产交期。

**英文回复草稿**
Hello, thank you for your inquiry. We can review our 500ml stainless steel vacuum bottles with one-color logo printing and standard individual paper boxes. Please share your logo file, preferred bottle style/color, and delivery terms and German destination port. We will then confirm the suitable model and quotation details. Lead time is subject to factory confirmation and will be estimated, not guaranteed.


In [11]:
print(first_result)

{'text': '**1. 已知需求**\n买家要500只不锈钢真空瓶，500ml，单色印公司logo，接受标准独立纸盒，交德国，时间“下个月”。未说明瓶型、材质等级、印刷位置、包装文字、贸易术语和目的港。\n\n**2. 初步建议和依据**\n可先按工厂现有500ml不锈钢真空瓶系列推荐，优先选起订量低、可单色丝印或移印的常规款；依据是买家接受标准纸盒且数量500只。报价需列明：瓶体、单色logo印刷、标准独立纸盒、常规外箱；未含运费、关税、德国目的港费用及可能的打样/版费。交期只能给“估计”，不能承诺下月到货；需先确认库存和排产。\n\n**3. 需要确认的问题**\n请确认具体瓶型/颜色、logo文件与印刷位置、是否要打样、贸易术语（FOB/CIF等）及德国目的港、外箱唛头要求。另需工厂确认500ml常规款500只的起订量、单色印刷费和当前排产交期。\n\n**英文回复草稿**\nHello, thank you for your inquiry. We can review our 500ml stainless steel vacuum bottles with one-color logo printing and standard individual paper boxes. Please share your logo file, preferred bottle style/color, and delivery terms and German destination port. We will then confirm the suitable model and quotation details. Lead time is subject to factory confirmation and will be estimated, not guaranteed.', 'usage': {'prompt_tokens': 205, 'completion_tokens': 328, 'total_tokens': 533, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 205}, 'model': 'deepseek-flash', 'finish_reason': '

先读这次真实回答，记录它的表现：

- 它是否准确理解了数量、容量、Logo、包装和目的地？
- 它是否承认缺少工厂资料，或者追问可提供的产品和价格？
- 如果它说“有货”“可以按时送到”或给出确定价格，这些信息从哪里来？假设举例有没有明确标出？

**模型可能第一次就很谨慎。** 那也是有效观察：谨慎能避免乱承诺，但它还不能据此指出我们哪款杯子合适。我们不会预先断言第一版一定出错。

**我的观察：**（引用这次回答中的一句话，说明它知道什么、还不知道什么。）


## 3 · 把工厂资料拿出来

刚才缺的是这家工厂自己的事实。我们已经准备了 [factory_brief.md](data/factory_brief.md)，现在读出来。下面的 `display(Markdown(...))` 只负责把文件显示成易读的表格，不会请求模型。


In [12]:
factory_brief = (course_dir / "data" / "factory_brief.md").read_text(encoding="utf-8")
display(Markdown(factory_brief))


# 山行杯业：给销售助手的工厂资料

**资料性质：全部为课程模拟。** 山行杯业、商品、库存、价格和交期均为教学设定，不对应真实商家，也不代表保温杯行业的通用规则。模拟资料版本：2026-10-31；本课各次实验都使用同一份库存快照，询盘本身不会扣减库存。

## 我们卖什么

我们是一家虚构的中国保温杯出口工厂，面向海外企业采购客户。下表商品均为不锈钢保温杯；其他材质细节、保温性能、认证和检测报告尚未提供，不能自行补充。

每个 SKU 是一个确定的商品规格和颜色。库存是尚未印 Logo 的杯子数量，不能把其他颜色的库存直接当成本款库存。价格币种为美元（USD），按同一订单中**每个 SKU 的数量**分别选档；不同 SKU 的数量不能合并选低价。

| SKU | 容量 | 颜色 | 可用库存（个） | 100–499 个：USD/个 | 500–999 个：USD/个 | 1000 个及以上：USD/个 |
|---|---:|---|---:|---:|---:|---:|
| S350-W | 350ml | 白色 | 400 | 3.80 | 3.50 | 3.20 |
| S350-B | 350ml | 黑色 | 900 | 3.80 | 3.50 | 3.20 |
| S500-W | 500ml | 白色 | 800 | 4.20 | 3.90 | 3.60 |
| S500-B | 500ml | 蓝色 | 200 | 4.60 | 4.20 | 3.90 |
| S600-S | 600ml | 银色 | 600 | 4.80 | 4.40 | 4.00 |
| S600-B | 600ml | 黑色 | 300 | 4.80 | 4.40 | 4.00 |
| S750-S | 750ml | 银色 | 1200 | 5.20 | 4.80 | 4.40 |
| S1000-B | 1000ml | 黑色 | 100 | 6.20 | 5.80 | 5.40 |

## 数量、Logo 和包装

- 不印 Logo 时，每个 SKU 最少订购 100 个。表中的单价仅包含杯子和普通独立纸盒包装。
- 印 Logo 时，每个 SKU 最少订购 500 个，且使用同一图案。我们目前只提供单色印刷；需要客户确认图案、印刷颜色和位置，并提供可用的矢量文件。
- 印刷费为 USD 0.25/个；每个订单另外收取制版费 USD 30。本课一个订单只使用一个 Logo 图案。
- 普通包装为每杯一个无品牌纸盒。礼盒、彩盒和其他包装尚无报价与交期，不能默认免费或当天可做。
- 不提供未记录的折扣、最低数量例外或加急服务。若客户不能接受现有条件，可以提出待客户确认的调整方向，不能说工厂已批准。

## 货物什么时候能发出

下列时间都使用**日历天**。计时从订单确认、付款完成且所需 Logo 文件获双方确认后开始；客户首次发来询盘不等于计时开始。

- 库存足够且不印 Logo：整理、检查和普通包装需要 2 天。
- 库存足够且印 Logo：加工、检查和普通包装合计需要 12 天。现货仍是没有 Logo 的杯子，不能跳过这 12 天。
- 本款库存不足：先补货 20 天，再进行上述 2 天或 12 天处理。本课先等整单货齐，再统一处理和发出；分批交付没有现成规则。
- 库存仅是当前快照。实际接单前仍需复核，不能因为客户询问了商品就声称已经为其锁定库存。

## 运往德国

课程暂设一种普通运输方式：从货物交给承运商起，运输参考为 8–12 天。这个估计需要完整收货地址、邮编和承运商可服务性确认；额外清关停留无法估算，不包含在 8–12 天中。没有其他运输方式的资料。

现有资料**没有运费、税费或进口费用**。可以说明已知的商品与定制费用，并列出尚未包含的项目；不能把它叫作送到德国的最终总价。若客户给出预算，需要确认预算包含哪些费用。

## 收到询盘时要确认什么

先确认数量、容量、颜色、是否印 Logo、包装要求、完整收货地址与明确的到货日期。“下个月”没有说明哪一天，不能据此保证交付。Logo 文件和印刷细节也会影响何时具备开工条件。

按资料可以计算条件满足后的最早估计时间，但不能把参考时间写成已经确认的到货承诺。资料没有的信息应明确标为待确认；可用的方案也应说明成立条件。我们的交付物是销售人员可核对的建议和回复草稿，尚未向买家发送，也未创建订单。


先亲自看三件事：

1. 哪个 500ml 型号有足够库存供这次 500 个订单使用？
2. 有现货，是否就意味着印 Logo 的订单今天能发货？
3. 资料有没有给出运费，以及一个已确认的德国收货地址？

注意：文件现在只被 Python 读到了。**模型不会因为文件放在你的电脑上，就自动看见它。** 我们需要把资料文字放进这次发出的消息里。


### TODO 1 · 把判断依据加入请求

任务要求和买家原话都保持不变，只增加工厂资料，这样更容易比较资料带来的变化。

**要用的写法：字符串相加。** `"说明" + "\n\n" + "资料"` 会得到一段包含说明、空行和资料的文字；`\n` 表示换行。把下面的 `None` 换成用 `task_rules`、空行和 `factory_brief` 拼出的文字。

这里资料由我们维护，因此可以和任务说明一起放入 `system`。买家原话仍单独放入 `user`。身份标签有助于表达边界，但本身不能保证模型一定按规则判断，回答仍需要核对。


<details><summary>提示 1：模型少了什么？</summary>

它已经拿过任务要求；这次要同时看到工厂商品与经营规则。

</details>
<details><summary>提示 2：需要哪两个变量？</summary>

任务在 `task_rules`，整页资料在 `factory_brief`，中间加一个空行。

</details>
<details><summary>代码答案</summary>

```python
with_facts_rules = task_rules + "\n\n" + factory_brief
```

</details>


In [14]:
with_facts_rules = task_rules + "\n\n" + factory_brief # TODO 1：拼接任务说明、空行和工厂资料
assert isinstance(with_facts_rules, str), "先完成 TODO 1，再继续。"
assert factory_brief in with_facts_rules and task_rules in with_facts_rules, "这次请求要同时包含任务说明和工厂资料。"
with_facts_messages = [
    {"role": "system", "content": with_facts_rules},
    {"role": "user", "content": buyer_text},
]
print("资料已放进即将发送的消息；本格还未发请求。")


资料已放进即将发送的消息；本格还未发请求。


**下面发出第 2 次真实请求。** 没有更换模型、任务要求或买家原话，输入里增加了工厂资料。保存为新变量，以后仍能查看第一次回答。


In [15]:
with_facts_result = call_model(with_facts_messages, config)
print(with_facts_result["text"])


## 1. 已知需求

- 数量：500 个；容量：500ml；不锈钢保温杯。
- 需印单色公司 Logo，普通独立纸盒包装可接受。
- 目的地：德国；希望“下个月”到货，但未给具体日期。
- 未提供：颜色偏好、Logo 图案/颜色/位置/矢量文件、完整收货地址和邮编。

## 2. 初步建议和依据

- 可推荐 S500-W（白色，库存 800）或 S500-B（蓝色，库存 200）。两者 500ml 均满足；但印 Logo 每 SKU 最低 500 个，S500-B 库存仅 200，会触发补货 20 天，故优先建议 S500-W。
- 500 个落在 500–999 档：S500-W 为 USD 3.90/个；印刷费 USD 0.25/个；制版费 USD 30/单。报价含杯子和普通独立纸盒、单色 Logo 印刷；不含运费、税费、进口费用及礼盒等特殊包装。
- 交期：库存足够且印 Logo 为 12 天，从订单确认、付款完成、Logo 文件双方确认后起算；运输参考 8–12 天，需地址和承运商确认，不含清关停留。因此“下个月”无法承诺，需具体到货日期。

## 3. 需要确认的问题

1. 选白色还是蓝色？若选蓝色，是否接受补货时间？
2. Logo 图案、印刷颜色、位置及矢量文件？
3. “下个月”的具体到货日期？
4. 完整收货地址和邮编？
5. 报价是否按上述条件确认？

## 英文回复草稿

Dear Customer,

Thank you for your inquiry. For 500ml with a one-color logo, we suggest S500-W (white), 500 pcs. Quotation: USD 3.90/pc + USD 0.25/pc printing + USD 30 setup. This includes the bottle, standard individual paper box and one-color logo printing. Freight, taxes and import charges are not included.

With sufficient stock, logo production takes 12 days after order confirmatio

### 对照资料核对，而不是看回答听起来是否专业

| 要检查的地方 | 本课资料给出的依据 |
|---|---|
| 可选商品 | 白色 S500-W 的库存足够 500 个；蓝色 S500-B 的库存不够，需要补货 |
| 定制限制 | 500 个达到单色 Logo 的最低数量；现货也需要 12 个日历天完成加工和包装 |
| 价格范围 | 商品、Logo 加工和普通包装可以按规则估算；运费和税费没有给出 |
| 到货判断 | 德国运输参考 8–12 天，且要确认地址和渠道；“下个月”不是一个确定日期 |
| 需要追问 | 颜色、Logo 文件、完整地址、具体到货日期；英文回复应和中文建议一致 |

如果它选择白色并给出金额，商品与 Logo 的小计应是 **500 × 3.90 + 500 × 0.25 + 30 = 2105 美元**，普通纸盒已含。这不是送到德国的全包价。这里是按课程规则人工算出的核对值，后续课程再把计算交给 Python 工具。

它可以给有条件的建议，也可以先追问，措辞不必和某个标准答案完全一样。关键看事实、条件和未知项是否处理正确。比较它是否比第一次更具体；没有改善或出现新错误，也原样记下来。

**我的核对记录：**（模型原句 → 对应资料 → 正确、错误或仍待确认。）


## 4 · 买家改口：我 10 天内就要收到

一封询盘处理得好，还不能说明它会处理约束变化。现在保持工厂资料不变，只替换买家的这一次需求：500 个、500ml、Logo、普通包装都不变，把到货要求改为 **10 个日历天内**。

这是另一个独立请求。本课的函数只发送你传入的 `messages`，不会自动把以前的回答带过去。连续谈单和保存历史会在后面的课里做。


### TODO 2 · 让这一次请求使用新的买家原话

`inquiries` 是读自 JSON 文件的字典。`inquiries["initial"]` 取初始询盘，`inquiries["urgent"]` 取加急询盘。把下面的 `None` 换成取加急询盘的表达式，再先打印检查，避免把旧问题重新发一遍。


<details><summary>提示 1：这一轮只改什么？</summary>

工厂资料和工作要求都不变，改的是这次买家的原话。

</details>
<details><summary>代码答案</summary>

```python
urgent_text = inquiries["urgent"]
```

</details>


In [16]:
urgent_text = inquiries["urgent"]  # TODO 2：读取加急询盘
assert isinstance(urgent_text, str) and urgent_text == inquiries["urgent"], "请先读取 urgent 对应的买家原话。"
urgent_messages = [
    {"role": "system", "content": with_facts_rules},
    {"role": "user", "content": urgent_text},
]
print(urgent_text)


Hello, we are looking for 500 stainless steel vacuum bottles, 500ml each, with our company logo printed in one color. Your standard individual paper boxes are fine. We need the entire order delivered to Germany within 10 calendar days after order confirmation, payment, and approval of our print-ready logo file. Which products would you recommend, and could you send us a quotation?


**下面发出第 3 次真实请求。** 运行前先预测：现货也需要 12 天加工，再加运输，模型是否有依据答应 10 天到货？


In [17]:
urgent_result = call_model(urgent_messages, config)
print(urgent_result["text"])


## 1. 已知需求
- 500 个不锈钢保温杯，500ml，单色印 Logo，普通独立纸盒。
- 目的地德国；要求“订单确认、付款、Logo 文件确认后 10 个日历天内”整单交付。
- 需要推荐产品和报价。

## 2. 初步建议和依据
- 500ml 有 S500-W（白，库存 800）和 S500-B（蓝，库存 200）。印 Logo 每 SKU 最低 500 个，因此只有 S500-W 可满足 500 个同款同色；S500-B 库存不足，需补货 20 天，无法满足 10 天要求。
- 报价可按 S500-W 500 个：单价 USD 3.90/个，印刷费 USD 0.25/个，制版费 USD 30/单。以上含杯子和普通独立纸盒，不含运费、税费及进口费用，不是到德国最终总价。
- 交期：库存足够且印 Logo 需 12 天处理，已超过客户要求的 10 天；因此 10 天内交付无法承诺。运输参考 8–12 天，且不含清关停留，需完整地址和邮编确认。

## 3. 需要确认的问题
1. 是否接受 S500-W 白色，或可调整数量/颜色？
2. Logo 矢量文件、印刷颜色和位置是否已确定？
3. 完整收货地址、邮编及“10 天”是指发货还是到货？
4. 预算是否含运费和税费？

## 英文回复草稿
Dear [Name],

Thank you for your inquiry. For 500ml with a one-color logo, we recommend S500-W (white), as 500 pieces meet the logo minimum. Quotation for 500 pieces: USD 3.90/piece + USD 0.25/piece printing + USD 30 setup. This includes cups and standard individual paper boxes, excluding freight, taxes and import charges.

Please note logo production takes 12 calendar days after order confirmation, payment and logo approval, so 10-day 

### TODO 3 · 由你判断这次接单建议是否站得住

这次不写新代码。对照实际回答，在下面填“通过 / 未通过 / 无法判断”，并引用一句证据。

| 核对项 | 我的判断 | 模型原句或遗漏 |
|---|---|---|
| 明确识别 10 天到货与现有规则冲突 | 通过 | 待填写 |
| 没有因为有库存就跳过 Logo 加工时间 | 待填写 | 待填写 |
| 没有擅自发明加急生产、航空运费或更快渠道 | 待填写 | 待填写 |
| 如果提出替代方案，说明还需要客户或工厂确认 | 待填写 | 待填写 |
| 英文回复没有推翻中文分析、偷偷承诺按时到货 | 待填写 | 待填写 |

人工核对依据：现货定制按资料最短估计 **12 + 8 = 20 个日历天**，且运输仍有条件，因此本课资料不支持承诺 10 天到货。可以建议协商日期，或询问是否愿意调整需求；提出选项不等于选项已经被确认可行。

**假如模型出错了**：保留回答，写出违反哪条资料。无需不停重新运行，直到碰巧得到满意回答。一次正确只说明这一次通过，后面会用更多未见过的询盘检查稳定性。


## 5 · 看清这三次回答从哪里来

下面只读取已经得到的结果，不再请求模型。`model` 是服务返回的模型名称，`usage` 是服务返回的用量，`elapsed_seconds` 是客户端测得的一次请求耗时，包含网络和服务处理时间。

不同服务可能不返回完整用量，工具会保留实际返回值；缺失就显示缺失，不用手填数字冒充实测。本课不写死价格，费用以你所用服务和模型的计费规则为准。


In [18]:
for label, result in [("只有询盘", first_result), ("加入资料", with_facts_result), ("要求10天到货", urgent_result)]:
    print(label)
    print("模型：", result["model"], "结束原因：", result["finish_reason"])
    print("用量：", result["usage"], "请求耗时（秒）：", result["elapsed_seconds"])


只有询盘
模型： deepseek-flash 结束原因： stop
用量： {'prompt_tokens': 205, 'completion_tokens': 328, 'total_tokens': 533, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 205} 请求耗时（秒）： 3.065
加入资料
模型： deepseek-flash 结束原因： stop
用量： {'prompt_tokens': 1317, 'completion_tokens': 536, 'total_tokens': 1853, 'prompt_cache_hit_tokens': 128, 'prompt_cache_miss_tokens': 1189} 请求耗时（秒）： 3.383
要求10天到货
模型： deepseek-flash 结束原因： stop
用量： {'prompt_tokens': 1336, 'completion_tokens': 489, 'total_tokens': 1825, 'prompt_cache_hit_tokens': 1152, 'prompt_cache_miss_tokens': 184} 请求耗时（秒）： 3.56


## 把今天的实验解释清楚

前面学的 logits、softmax 描述模型怎样选择接下来生成的内容。现在模型会根据更长的输入生成回答，但“能接着写”并不自动等于“知道这家工厂的现状”。加入工厂资料，是为**本次回答提供上下文**，没有重新训练模型，也没有让这份资料永久进入模型权重。

你今天实际完成的是：**本地读取询盘 → 准备消息 → 调用真实模型 → 对照工厂资料核对建议。** 建议还没有发送给买家，也没有变成真实订单。

试着用自己的话回答：

1. 哪些资料是我们模拟的，哪些结果是这次模型生成的？
2. 把文件读到 `factory_brief` 后，为什么还要把它放入 `messages`？
3. 如果第一版已经很谨慎，第二版增加资料的价值是什么？
4. 为什么“英文写得好”不能证明“可以接下这张订单”？
5. 哪一条观察说明下一步最需要增加什么能力？

今天一页资料可以全部发给模型。等商品目录和历史记录变多，我们会遇到“每次该找哪几条资料”的问题，下一课就从这里继续。算价工具、连续谈单和多步 Agent 都会沿着同一家工厂逐步加入。

做完可打开 [homework.ipynb](homework.ipynb)。课堂和作业的检查标准都来自事先写好的规则；模拟验证不能替代真实商家需求验证。
